In [1]:
import pandas as pd
import os
import numpy as np
import folium
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages
from folium.plugins import FastMarkerCluster
from folium.plugins import MarkerCluster
from pathlib import Path
from datetime import datetime
from pyproj import Transformer

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

In [2]:
# Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()

# Pfad zu "Komplette_Datenbank" (eins zurück, dann in die Struktur)
db_root = (
    base_dir.parent 
    / "1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken"
    / "Komplette_Datenbank"
)

# Alle Unterordner holen, Ordner mit "aktuellstem" Zeitstempel setzen
timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)

# Komplette Datenbank als CSV
database_path = latest_folder / "Komplette_Datenbank.csv"

print("Verwendeter Datenbankpfad:")
print(database_path)

Verwendeter Datenbankpfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Komplette_Datenbank\2025-11-23_11-44-33\Komplette_Datenbank.csv


In [3]:
# Ordner "Datenanalyse"
analysis_root = base_dir / "Datenanalyse"
analysis_root.mkdir(exist_ok=True)

# Zeitstempel erzeugen
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Neuen Ordner mit Zeitstempel erzeugen
output_dir = analysis_root / timestamp
output_dir.mkdir(parents=True, exist_ok=False)

print("Erzeugter Output-Ordner:")
print(output_dir)

Erzeugter Output-Ordner:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Explorative-Datenanalyse\Datenanalyse\2025-11-24_22-48-51


In [9]:
# ============================================================
# PDF-REPORT: Datenverteilung der kompletten Datenbank
# Baut DIREKT auf deinem Path-Setup (database_path, output_dir) auf.
#
# Output:
#   Seite 1: Balkendiagramm "nicht-leere Werte pro Spalte"
#   Seite 2: Eckdaten + Vollständigkeit in 10%-Schritten als Anteile
#   Seite 3+: pro Spalte eine Seite:
#       - numerisch: Histogramm (sinnvolle x-Skala via 1–99% Quantile) + Boxplot
#       - datetime: Zeit-Hist + Counts/Jahr
#       - kategorisch: Balken Top 20 + Pie Top vs Rest
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
# 1) DEIN PATH-SETUP (unverändert)
# ------------------------------------------------------------

# Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()

# Pfad zu "Komplette_Datenbank" (eins zurück, dann in die Struktur)
db_root = (
    base_dir.parent
    / "1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken"
    / "Komplette_Datenbank"
)

# Alle Unterordner holen, Ordner mit "aktuellstem" Zeitstempel setzen
timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)

# Komplette Datenbank als CSV
database_path = latest_folder / "Komplette_Datenbank.csv"

print("Verwendeter Datenbankpfad:")
print(database_path)

# Ordner "Datenanalyse"
analysis_root = base_dir / "Datenanalyse"
analysis_root.mkdir(exist_ok=True)

# Zeitstempel erzeugen
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Neuen Ordner mit Zeitstempel erzeugen
output_dir = analysis_root / timestamp
output_dir.mkdir(parents=True, exist_ok=False)

print("Erzeugter Output-Ordner:")
print(output_dir)


# ------------------------------------------------------------
# 2) HELFERFUNKTIONEN
# ------------------------------------------------------------

def _try_force_numeric(s: pd.Series) -> pd.Series:
    """Erzwingt numerisch, wo möglich (Strings -> NaN falls nicht konvertierbar)."""
    return pd.to_numeric(s, errors="coerce")

def _try_force_datetime(s: pd.Series) -> pd.Series:
    """Erzwingt datetime, wo möglich."""
    # nur versuchen, wenn object/strings oder schon datetime
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    if s.dtype == "object":
        parsed = pd.to_datetime(s, errors="coerce")
        # nur akzeptieren wenn ein sinnvoller Anteil konvertierbar ist
        if parsed.notna().mean() > 0.5:
            return parsed
    return s

def _column_page(ax, col: str, s: pd.Series, sample_for_categoricals: int = 200000):
    """Erstellt eine Seite pro Spalte mit passendem Plot je Datentyp."""
    n_total = len(s)
    n_nonnull = int(s.notna().sum())
    n_null = n_total - n_nonnull

    # Figure-Layout
    ax.axis("off")
    fig = ax.figure
    fig.suptitle(f"Spalte: {col}", fontsize=16, y=0.98)

    info_text = (
        f"Gesamt: {n_total}\n"
        f"Nicht leer: {n_nonnull} ({n_nonnull/n_total:.1%})\n"
        f"Leer: {n_null} ({n_null/n_total:.1%})\n"
        f"Original-Typ: {s.dtype}"
    )
    ax.text(0.02, 0.92, info_text, va="top", ha="left",
            fontsize=11, family="monospace")

    # Achsenbereiche für Plots
    plot_ax1 = fig.add_axes([0.08, 0.10, 0.84, 0.35])
    plot_ax2 = fig.add_axes([0.08, 0.55, 0.84, 0.35])

    # -----------------------------
    # A) NUMERISCH?
    # -----------------------------
    s_num = _try_force_numeric(s)
    if s_num.notna().sum() > 0:
        s_clean = s_num.dropna()

        # sinnvolle X-Skala über Quantile (1–99%), damit Ausreißer nicht alles platt machen
        q1, q99 = s_clean.quantile([0.01, 0.99])
        if np.isfinite(q1) and np.isfinite(q99) and q1 < q99:
            x_min, x_max = q1, q99
        else:
            x_min, x_max = s_clean.min(), s_clean.max()

        # Freedman–Diaconis-Bins (automatisch)
        bins = "fd"

        # Histogramm
        plot_ax2.hist(s_clean, bins=bins, range=(x_min, x_max))
        plot_ax2.set_title("Numerische Verteilung (Histogramm)")
        plot_ax2.set_xlabel(col)
        plot_ax2.set_ylabel("Häufigkeit")

        # optional Log-Y bei sehr schiefer Verteilung
        med = s_clean.median()
        if med > 0 and s_clean.max() > 50 * med:
            plot_ax2.set_yscale("log")

        # Boxplot
        plot_ax1.boxplot(s_clean, vert=False)
        plot_ax1.set_title("Boxplot (bereinigte Skala)")
        plot_ax1.set_xlabel(col)
        plot_ax1.set_yticks([])

        return

    # -----------------------------
    # B) DATETIME?
    # -----------------------------
    s_dt = _try_force_datetime(s)
    if pd.api.types.is_datetime64_any_dtype(s_dt) and s_dt.notna().sum() > 0:
        s_dt_clean = s_dt.dropna()

        plot_ax2.hist(s_dt_clean.view("int64"), bins=30)
        plot_ax2.set_title("Zeitverteilung (Histogramm)")
        plot_ax2.set_ylabel("Häufigkeit")

        try:
            by_year = s_dt_clean.dt.to_period("Y").value_counts().sort_index()
            plot_ax1.bar(by_year.index.astype(str), by_year.values)
            plot_ax1.set_title("Counts pro Jahr")
            plot_ax1.set_ylabel("Häufigkeit")
            for label in plot_ax1.get_xticklabels():
                label.set_rotation(45)
                label.set_ha("right")
        except Exception:
            plot_ax1.text(0.5, 0.5, "Keine sinnvolle Jahres-Aggregation",
                          ha="center", va="center")
        return

    # -----------------------------
    # C) KATEGORISCH / TEXT
    # -----------------------------
    s_cat = s.dropna().astype(str)
    if len(s_cat) > 0:
        if len(s_cat) > sample_for_categoricals:
            s_cat = s_cat.sample(sample_for_categoricals, random_state=0)

        vc = s_cat.value_counts()
        top_n = 20
        vc_top = vc.head(top_n)

        plot_ax2.bar(vc_top.index, vc_top.values)
        plot_ax2.set_title(f"Kategorische Verteilung (Top {top_n})")
        plot_ax2.set_xlabel("Kategorie")
        plot_ax2.set_ylabel("Häufigkeit")
        plot_ax2.tick_params(axis="x", rotation=45)

        rest = vc.iloc[top_n:].sum()
        if rest > 0:
            plot_ax1.pie(
                [vc_top.sum(), rest],
                labels=[f"Top {top_n}", "Rest"],
                autopct=lambda p: f"{p:.1f}%"
            )
            plot_ax1.set_title("Top vs. Rest")
        else:
            plot_ax1.text(0.5, 0.5, "Keine weiteren Kategorien",
                          ha="center", va="center")
        return

    # -----------------------------
    # D) KOMPLETT LEER
    # -----------------------------
    plot_ax1.text(0.5, 0.5, "Keine Werte", ha="center", va="center")
    plot_ax2.text(0.5, 0.5, "–", ha="center", va="center")


# ------------------------------------------------------------
# 3) PDF-GENERATOR
# ------------------------------------------------------------

def create_data_distribution_pdf_from_paths(
    database_path: Path,
    output_dir: Path,
    pdf_name: str = "Komplette_Datenbank_Distribution.pdf",
    sample_for_categoricals: int = 200000
) -> Path:

    database_path = Path(database_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = output_dir / pdf_name

    if not database_path.exists():
        raise FileNotFoundError(f"Input-Datei existiert nicht: {database_path}")

    print("⚙️ Starte PDF-Erstellung...")
    print(f"📥 Lade Datenbank: {database_path}")

    df = pd.read_csv(database_path, low_memory=False)

    n_rows, n_cols = df.shape
    print(f"✅ Daten geladen: {n_rows} Zeilen, {n_cols} Spalten")

    nonnull_counts = df.notna().sum().sort_values(ascending=False)

    # Vollständigkeit je Zeile in %
    row_comp = df.notna().mean(axis=1) * 100
    complete_rows = int((row_comp == 100).sum())

    # 10%-Bins (0–10–20–...–100)
    bins = np.arange(0, 110, 10)
    labels = [f"{bins[i]}–{bins[i+1]}%" for i in range(len(bins)-1)]
    binned = pd.cut(row_comp, bins=bins, labels=labels, include_lowest=True)
    bin_counts = binned.value_counts().reindex(labels, fill_value=0)
    bin_percent = (bin_counts / n_rows * 100).round(2)

    print(f"📝 Schreibe PDF nach: {pdf_path}")

    with PdfPages(pdf_path) as pdf:

        # ---------------- Seite 1: Nicht-leere Werte je Spalte ----------------
        print("📈 Seite 1: Spalten-Vollständigkeit")
        fig, ax = plt.subplots(figsize=(11.7, 8.3))
        ax.bar(nonnull_counts.index.astype(str), nonnull_counts.values)
        ax.set_title("Anzahl nicht-leerer Werte pro Spalte")
        ax.set_xlabel("Spalte")
        ax.set_ylabel("Nicht-leere Werte")
        ax.set_ylim(0, n_rows)
        ax.tick_params(axis="x", rotation=60)
        ax.grid(axis="y", alpha=0.3)
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # ---------------- Seite 2: Eckdaten + Vollständigkeit ----------------
        print("📄 Seite 2: Eckdaten / Zeilen-Vollständigkeit")
        fig, ax = plt.subplots(figsize=(8.3, 11.7))
        ax.axis("off")
        ax.set_title("Eckdaten / Zeilen-Vollständigkeit", fontsize=16, pad=20)

        txt = (
            f"Zeilen: {n_rows}\n"
            f"Spalten: {n_cols}\n"
            f"Vollständige Zeilen (100% befüllt): {complete_rows} "
            f"({complete_rows/n_rows:.2%})\n\n"
            "Verteilung der Zeilen-Vollständigkeit (10%-Schritte):\n"
        )
        for l in labels:
            txt += f"• {l}: {bin_percent[l]}% aller Datenzeilen\n"

        ax.text(0.05, 0.95, txt, ha="left", va="top",
                fontsize=12, family="monospace")

        pdf.savefig(fig)
        plt.close(fig)

        # ---------------- Seite 3+: je Spalte ----------------
        print("📚 Spaltenseiten...")
        for i, col in enumerate(df.columns, start=1):
            print(f"   → {i}/{len(df.columns)}: {col}")
            fig, ax = plt.subplots(figsize=(8.3, 11.7))
            _column_page(ax, col, df[col], sample_for_categoricals=sample_for_categoricals)
            # KEIN tight_layout hier wegen add_axes -> verhindert Warnungen/Clipping
            pdf.savefig(fig)
            plt.close(fig)

    print(f"✅ PDF fertig: {pdf_path.resolve()}")
    return pdf_path


# ------------------------------------------------------------
# 4) AUFRUF (WICHTIG: außerhalb der Funktion!)
# ------------------------------------------------------------
pdf_path = create_data_distribution_pdf_from_paths(
    database_path=database_path,
    output_dir=output_dir,
    pdf_name="Komplette_Datenbank_Distribution.pdf"
)

print("PDF gespeichert unter:", pdf_path)


Verwendeter Datenbankpfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Komplette_Datenbank\2025-11-23_11-44-33\Komplette_Datenbank.csv
Erzeugter Output-Ordner:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Explorative-Datenanalyse\Datenanalyse\2025-11-24_22-51-46
⚙️ Starte PDF-Erstellung...
📥 Lade Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Komplette_Datenbank\2025-11-23_11-44-33\Komplette_Datenbank.csv
✅ Daten geladen: 173091 Zeilen, 30 Spalten
📝 Schreibe PDF nach: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Explorative-Datenanalyse\Datenanalyse\2025-11-24_22-51-46\Komplette_Datenbank_Distribution.pdf
📈 Seite 1: Spalten-Vollständigkeit
📄 Seite 2:

C:\Users\lucca\AppData\Local\Temp\ipykernel_20896\2297864485.py:75: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s, errors="coerce")


   → 2/30: well_or_spring_name
   → 3/30: WGS84_latitude
   → 4/30: WGS84_longitude
   → 5/30: depth_bgl_in_m
   → 6/30: rock_type
   → 7/30: stratigraphic_period


C:\Users\lucca\AppData\Local\Temp\ipykernel_20896\2297864485.py:75: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s, errors="coerce")


   → 8/30: temperature_in_c
   → 9/30: electrical_conductivity_25c_in_uS/cm
   → 10/30: pH
   → 11/30: redox_potential_in_mV
   → 12/30: total_dissolved_solids_in_mmol/L
   → 13/30: O2_in_mmol/L
   → 14/30: Na_in_mmol/L
   → 15/30: Mg_in_mmol/L
   → 16/30: Ca_in_mmol/L
   → 17/30: Cl_in_mmol/L
   → 18/30: SO4_in_mmol/L
   → 19/30: HCO3_in_mmol/L
   → 20/30: Li_in_mmol/L
   → 21/30: K_in_mmol/L
   → 22/30: Sr_in_umol/L
   → 23/30: NH4_in_umol/L
   → 24/30: Fe_in_mmol/L
   → 25/30: Mn_in_mmol/L
   → 26/30: F_in_umol/L
   → 27/30: NO3_in_mmol/L
   → 28/30: H2SiO3_in_umol/L
   → 29/30: HS_in_mmol/L
   → 30/30: Database_number
✅ PDF fertig: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Explorative-Datenanalyse\Datenanalyse\2025-11-24_22-51-46\Komplette_Datenbank_Distribution.pdf
PDF gespeichert unter: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Explorative-Datenanalyse\Daten